# RNN + spaCy-Embeddings an.

In [2]:
import spacy
import torch
import torch.nn as nn
import torch.optim as optim

# -----------------------------
# 1. Vorbereitung
# -----------------------------
nlp = spacy.load("de_core_news_sm")
torch.manual_seed(42)

# Trainingsdaten (wie zuvor)
pos_texts = [
    "Das ist super!", "Ich liebe es.", "Einfach fantastisch.", "Sehr gut gemacht.", "Ich bin begeistert.",
    "Wunderbare Arbeit.", "Klasse Leistung.", "Absolut empfehlenswert.", "Ein echtes Highlight.", "Top Qualität.",
    "Sehr hilfreich.", "Ich bin sehr zufrieden.", "Großartig!", "Beste Entscheidung.", "Es macht Spaß.",
    "Perfekt gelaufen.", "Toller Service.", "Sehr freundlich.", "Beeindruckend.", "Gerne wieder.",
    "Alles bestens.", "Einwandfrei.", "Hervorragend.", "Spitzenklasse.", "Einfach nur toll."
]

neg_texts = [
    "Das ist schrecklich.", "Ich hasse es.", "Ganz furchtbar.", "Sehr schlecht.", "Ich bin enttäuscht.",
    "Miese Qualität.", "Nicht zu gebrauchen.", "Verschwendung von Zeit.", "Ein totaler Reinfall.", "Unterirdisch.",
    "Überhaupt nicht hilfreich.", "Ich bin unzufrieden.", "Grauenhaft!", "Fehlkauf.", "Es macht keinen Sinn.",
    "Viel zu teuer.", "Schlechter Service.", "Sehr unfreundlich.", "Enttäuschend.", "Nie wieder.",
    "Alles kaputt.", "Mangelhaft.", "Ungenügend.", "Katastrophe.", "Einfach nur mies."
]

train_texts = pos_texts + neg_texts
labels = torch.tensor([1]*25 + [0]*25)

# -----------------------------
# 2. RNN-Klassifikator
# -----------------------------
class RNNClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, num_layers, num_classes):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x):
        # x: (batch, seq_len, embedding_dim)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim)
        out, _ = self.rnn(x, h0)  # out: (batch, seq_len, hidden_dim)
        out = out[:, -1, :]       # letztes Hidden State
        out = self.fc(out)
        return out

# Hyperparameter
embedding_dim = 96
hidden_dim = 32
num_layers = 1
num_classes = 2

model = RNNClassifier(embedding_dim, hidden_dim, num_layers, num_classes)

# -----------------------------
# 3. Trainingsdaten vorbereiten (Token-Embeddings)
# -----------------------------
def text_to_tensor(text):
    doc = nlp(text)
    # Jedes Token als Embedding
    embeddings = torch.stack([torch.tensor(token.vector) for token in doc])
    return embeddings

# Pad sequences auf gleiche Länge
from torch.nn.utils.rnn import pad_sequence

train_sequences = [text_to_tensor(t) for t in train_texts]
train_sequences_padded = pad_sequence(train_sequences, batch_first=True)  # (batch, max_seq_len, embedding_dim)

# -----------------------------
# 4. Training
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

model.train()
for epoch in range(50):
    optimizer.zero_grad()
    outputs = model(train_sequences_padded)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# -----------------------------
# 5. Test-Sätze
# -----------------------------
tests = [
    "Der Kurs ist nicht schlecht.",
    "Der Service ist okay.",
    "Das Produkt ist teuer.",
    "Ich bin überrascht wie gut das ist.",
    "Der Film war langweilig aber schön gefilmt."
]

model.eval()
with torch.no_grad():
    for test_text in tests:
        seq = text_to_tensor(test_text).unsqueeze(0)  # batch=1
        # Pad auf Trainingslänge
        seq = nn.functional.pad(seq, (0,0,0,train_sequences_padded.size(1)-seq.size(1)))
        output = model(seq)
        probs = torch.softmax(output, dim=1)
        pred_class = torch.argmax(probs, dim=1).item()
        label_str = "Positiv" if pred_class == 1 else "Negativ"
        print(f"\nText: {test_text}")
        print(f"Wahrscheinlichkeiten (Negativ, Positiv): {probs.numpy()}")
        print(f"Predicted Class: {label_str}")

Epoch 10, Loss: 0.2383
Epoch 20, Loss: 0.0122
Epoch 30, Loss: 0.0007
Epoch 40, Loss: 0.0003
Epoch 50, Loss: 0.0001

Text: Der Kurs ist nicht schlecht.
Wahrscheinlichkeiten (Negativ, Positiv): [[0.80297077 0.19702926]]
Predicted Class: Negativ

Text: Der Service ist okay.
Wahrscheinlichkeiten (Negativ, Positiv): [[0.09316327 0.90683675]]
Predicted Class: Positiv

Text: Das Produkt ist teuer.
Wahrscheinlichkeiten (Negativ, Positiv): [[0.27918878 0.72081125]]
Predicted Class: Positiv

Text: Ich bin überrascht wie gut das ist.
Wahrscheinlichkeiten (Negativ, Positiv): [[0.5187562 0.4812438]]
Predicted Class: Negativ

Text: Der Film war langweilig aber schön gefilmt.
Wahrscheinlichkeiten (Negativ, Positiv): [[0.02136877 0.97863126]]
Predicted Class: Positiv
